# 🔍 Search Dork Generator
### For Artists, Designers & Researchers

---

### Copyright notice
HSLU DFK | Colabor: Zoom In Zoom Out | 2026 Thomas Knüsel, [MIT License](LICENSE). 
Feel free to modify and reuse the content. 

---

## What is "dorking" and why does it matter?

**Search dorking** (or Google dorking) refers to using advanced search operators — special commands like `site:`, `filetype:`, `intitle:`, or `intext:` — to construct highly precise queries that surface results ordinary searches miss. The term originally came from security research, where professionals used these techniques to find exposed data on the web, but today dorking is a standard skill for researchers, journalists, designers, and anyone who needs to go deeper than a keyword search.

For artists and designers this is especially useful: you can search museum collection databases for works by medium or period, find open-access research on specific movements, locate high-resolution image archives, or filter datasets by license type.

---

## ⚖️ Ethics & responsible use

Search operators are powerful and, like any tool, come with responsibilities:

- **Respect terms of service.** Most search engines permit manual dorking but prohibit automated scraping. Generating a query string in this notebook is fine; building a bot to fire thousands of requests is not.
- **Don't use dorking to access data you're not authorised to see.** The fact that something is technically findable does not mean it's meant to be accessed.
- **Mind privacy.** Operators that surface personal information (names, emails, addresses) should be used only for legitimate research, journalism, or your own data.
- **Cite your sources.** If your research leads to data or images, respect copyright and licensing — many databases in this notebook are open access but still require attribution.

---

## How to use this notebook

1. **Run all cells in order** — click *Kernel → Restart & Run All*, or press `Shift+Enter` on each cell from top to bottom.
2. The interactive UI appears at the bottom after the last cell.
3. **Select an engine** from the dropdown — each group (General, Privacy, Specialist, Social Media, Academic, Art & Design, Datasets) shows relevant operator fields.
4. **Fill in your phrase** and any operator fields you need.
5. Click **⚡ Generate Dorks** to build the query for your chosen engine, or **🌐 Compare All Engines** to see results across all platforms at once.
6. Each result row shows the query string and an **Open ↗** link that fires the search directly in your browser.

---

**Groups covered (8 total, 62 engines):**

| Group | Engines |
|---|---|
| General | Google · Bing · DuckDuckGo · Yandex · Baidu · Yahoo |
| Privacy | Brave · Ecosia · Qwant · Kagi |
| Specialist | SearXNG · GitHub · GitLab · Shodan · Internet Archive |
| Social Media | YouTube · Reddit · Twitter/X · Instagram · TikTok · VK · LinkedIn |
| Academic | arXiv · Google Scholar · PubMed · Semantic Scholar · BASE · CORE · Academia.edu · ResearchGate · JSTOR · OpenAlex · EuropePMC · DOAJ · bioRxiv · NASA ADS · SSRN |
| Media Art | Ars Electronica · ZKM · Rhizome · EAI · LIMA · Transmediale · HEK Basel · V2_ |
| Art & Design | Europeana · MoMA · V&A · Rijksmuseum · Cooper Hewitt · Google Arts & Culture · Behance · Artsy · Tate · The Met · Getty Museum · Louvre · Art Institute Chicago · National Gallery London · Smithsonian |
| Media Art | Rhizome · ZKM · Ars Electronica · EAI · LIMA · V2_ · transmediale · Vimeo |
| Datasets | Kaggle · Google Dataset Search · Zenodo · Hugging Face · Harvard Dataverse · OpenML · data.world · Papers with Code |


## Cell 1 — Install dependencies

This cell checks whether `ipywidgets` is installed and installs it if missing. `ipywidgets` provides the interactive dropdown menus, text fields, checkboxes, and buttons used throughout the UI. Run this once; subsequent runs will skip the install.

In [1]:
import importlib, subprocess, sys
for pkg in ['ipywidgets']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ Dependencies ready')

✅ Dependencies ready


## Cell 2 — Engine definitions

This cell defines every search engine in the notebook: its group, privacy rating, index type, a short note, an optional link to official operator documentation, base search URLs (web, images, video, news), and the inline operator templates used to build query strings. It also defines the filetype category list and a set of flags that control which UI sections are hidden for each engine.

In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML
import urllib.parse

ENGINES = {

    'Google': {
        'group':'General','privacy':'🔴 Low','index':'Own',
        'note':'Most comprehensive index. cache: removed 2024, related: removed 2023.',
        'docs':'https://support.google.com/websearch/answer/2466433',
        'base_url':'https://www.google.com/search?q=',
        'image_search':'https://www.google.com/search?tbm=isch&q=',
        'video_search':'https://www.google.com/search?tbm=vid&q=',
        'news_search':'https://www.google.com/search?tbm=nws&q=',
        'filetype':'filetype:{value}','site':'site:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','intext':'intext:{value}',
        'before':'before:{value}','after':'after:{value}',
        'imagesize':'imagesize:{value}','define':'define:{value}',
        'supports':['filetype','site','inurl','intitle','intext','before','after',
                    'imagesize','define','NOT','OR','exact','wildcard','numrange'],
    },
    'Bing': {
        'group':'General','privacy':'🔴 Low','index':'Own',
        'note':'Powers Ecosia & parts of DDG. inbody: ≈ intext:. contains: finds linked filetypes.',
        'docs':'https://support.microsoft.com/en-us/topic/advanced-search-options-b92e25f1-0085-4271-bdf9-14aaea720930',
        'base_url':'https://www.bing.com/search?q=',
        'image_search':'https://www.bing.com/images/search?q=',
        'video_search':'https://www.bing.com/videos/search?q=',
        'news_search':'https://www.bing.com/news/search?q=',
        'filetype':'filetype:{value}','site':'site:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','inbody':'inbody:{value}',
        'contains':'contains:{value}','language':'language:{value}',
        'supports':['filetype','site','inurl','intitle','inbody','contains',
                    'language','NOT','OR','exact','wildcard','numrange'],
    },
    'DuckDuckGo': {
        'group':'General','privacy':'🟢 High','index':'Bing + own',
        'note':'No tracking, no IP logging. Queries anonymised before Bing. !bangs for 13k+ sites.',
        'docs':'https://duckduckgo.com/duckduckgo-help-pages/results/syntax/',
        'base_url':'https://duckduckgo.com/?q=',
        'image_search':'https://duckduckgo.com/?iax=images&ia=images&q=',
        'video_search':'https://duckduckgo.com/?iax=videos&ia=videos&q=',
        'news_search':'https://duckduckgo.com/?ia=news&q=',
        'filetype':'filetype:{value}','site':'site:{value}',
        'inurl':'inurl:{value}','intitle':'intitle:{value}',
        'supports':['filetype','site','inurl','intitle','NOT','OR','exact','wildcard'],
    },
    'Yandex': {
        'group':'General','privacy':'🔴 Low','index':'Own (largest non-EN)',
        'note':'4th largest engine globally. Best for Cyrillic content. host: = exact domain; mime: = filetype.',
        'docs':'https://yandex.com/support/search/en/query-language/qlanguage',
        'base_url':'https://yandex.com/search/?text=',
        'image_search':'https://yandex.com/images/search?text=',
        'video_search':'https://yandex.com/video/search?text=',
        'news_search':'https://news.yandex.com/yandsearch?text=',
        'site':'site:{value}','host':'host:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','mime':'mime:{value}','lang':'lang:{value}',
        'supports':['site','host','inurl','intitle','mime','lang','NOT','OR','exact'],
    },
    'Baidu': {
        'group':'General','privacy':'🔴 Low','index':'Own (largest Chinese)',
        'note':'Dominant Chinese engine. site: filetype: intitle: inurl: work. No official English operator docs.',
        'docs':'',
        'base_url':'https://www.baidu.com/s?wd=',
        'image_search':'https://image.baidu.com/search/index?tn=baiduimage&word=',
        'filetype':'filetype:{value}','site':'site:{value}',
        'inurl':'inurl:{value}','intitle':'intitle:{value}',
        'supports':['filetype','site','inurl','intitle','NOT','OR','exact'],
    },
    'Yahoo': {
        'group':'General','privacy':'🔴 Low','index':'Bing-backed',
        'note':'Powered by Bing. Core Bing operators apply.',
        'docs':'https://help.yahoo.com/kb/SLN2194.html',
        'base_url':'https://search.yahoo.com/search?p=',
        'image_search':'https://images.search.yahoo.com/search/images?p=',
        'video_search':'https://video.search.yahoo.com/search/video?p=',
        'news_search':'https://news.search.yahoo.com/search?p=',
        'filetype':'filetype:{value}','site':'site:{value}',
        'inurl':'inurl:{value}','intitle':'intitle:{value}',
        'supports':['filetype','site','inurl','intitle','NOT','OR','exact','wildcard'],
    },

    'Brave': {
        'group':'Privacy','privacy':'🟢 High','index':'Own (independent)',
        'note':'Fully independent crawler, zero third-party index dependencies. intext: partial.',
        'docs':'https://search.brave.com/help/operators',
        'base_url':'https://search.brave.com/search?q=',
        'image_search':'https://search.brave.com/images?q=',
        'video_search':'https://search.brave.com/videos?q=',
        'news_search':'https://search.brave.com/news?q=',
        'filetype':'filetype:{value}','site':'site:{value}',
        'inurl':'inurl:{value}','intitle':'intitle:{value}',
        'supports':['filetype','site','inurl','intitle','NOT','OR','exact','wildcard'],
    },
    'Ecosia': {
        'group':'Privacy','privacy':'🟡 Medium','index':'Bing-backed',
        'note':'Plants trees with ad revenue. Berlin B-Corp. Bing operators apply; inbody: ≈ intext:. No dedicated operator docs.',
        'docs':'',
        'base_url':'https://www.ecosia.org/search?q=',
        'image_search':'https://www.ecosia.org/images?q=',
        'video_search':'https://www.ecosia.org/videos?q=',
        'news_search':'https://www.ecosia.org/news?q=',
        'filetype':'filetype:{value}','site':'site:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','inbody':'inbody:{value}','language':'language:{value}',
        'supports':['filetype','site','inurl','intitle','inbody','language','NOT','OR','exact','wildcard'],
    },
    'Qwant': {
        'group':'Privacy','privacy':'🟢 High','index':'Own + Bing',
        'note':'French, GDPR-native. intext: not officially documented.',
        'docs':'',
        'base_url':'https://www.qwant.com/?q=',
        'image_search':'https://www.qwant.com/?t=images&q=',
        'video_search':'https://www.qwant.com/?t=videos&q=',
        'news_search':'https://www.qwant.com/?t=news&q=',
        'filetype':'filetype:{value}','site':'site:{value}',
        'inurl':'inurl:{value}','intitle':'intitle:{value}',
        'supports':['filetype','site','inurl','intitle','NOT','OR','exact'],
    },
    'Kagi': {
        'group':'Privacy','privacy':'🟢 High','index':'Own + multiple',
        'note':'Paid subscription. No ads, no tracking. Full site/filetype/inurl/intitle/intext support.',
        'docs':'https://help.kagi.com/kagi/features/search-operators.html',
        'base_url':'https://kagi.com/search?q=',
        'image_search':'https://kagi.com/images?q=',
        'news_search':'https://kagi.com/news?q=',
        'filetype':'filetype:{value}','site':'site:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','intext':'intext:{value}',
        'before':'before:{value}','after':'after:{value}',
        'supports':['filetype','site','inurl','intitle','intext','before','after','NOT','OR','exact','wildcard'],
    },

    'SearXNG': {
        'group':'Specialist','privacy':'🟢 High','index':'Meta (aggregates all)',
        'note':'Open-source, self-hostable metasearch. Passes operators to underlying engines.',
        'docs':'https://docs.searxng.org/user/search-syntax.html',
        'base_url':'{instance}/search?q=',
        'image_search':'{instance}/search?categories=images&q=',
        'video_search':'{instance}/search?categories=videos&q=',
        'news_search':'{instance}/search?categories=news&q=',
        'filetype':'filetype:{value}','site':'site:{value}','inurl':'inurl:{value}',
        'intitle':'intitle:{value}','intext':'intext:{value}','language':'language:{value}',
        'supports':['filetype','site','inurl','intitle','intext','language','NOT','OR','exact','wildcard'],
    },
    'GitHub': {
        'group':'Specialist','privacy':'🟡 Medium','index':'Code only',
        'note':'Code search across public repos. Login required for full results.',
        'docs':'https://docs.github.com/en/search-github/searching-on-github',
        'base_url':'https://github.com/search?q=',
        'language':'language:{value}','user':'user:{value}','org':'org:{value}',
        'repo':'repo:{value}','path':'path:{value}','filename':'filename:{value}',
        'extension':'extension:{value}','size':'size:>{value}',
        'supports':['language','user','org','repo','path','filename','extension','size','NOT','OR','exact'],
    },
    'GitLab': {
        'group':'Specialist','privacy':'🟡 Medium','index':'Code only',
        'note':'Code search across public repos and snippets. Filter by scope.',
        'docs':'https://docs.gitlab.com/ee/user/search/advanced_search.html',
        'base_url':'https://gitlab.com/search?search=',
        'gl_group':'group:{value}','gl_project':'project:{value}',
        'gl_filename':'filename:{value}','gl_path':'path:{value}',
        'gl_extension':'extension:{value}','gl_language':'language:{value}',
        'supports':['gl_group','gl_project','gl_filename','gl_path','gl_extension','gl_language','NOT','OR','exact'],
    },
    'Shodan': {
        'group':'Specialist','privacy':'🟡 Medium','index':'Devices / IoT',
        'note':'Indexes internet-facing devices, open ports and service banners. Account required.',
        'docs':'https://help.shodan.io/the-basics/search-query-fundamentals',
        'base_url':'https://www.shodan.io/search?query=',
        'port':'port:{value}','country':'country:{value}','city':'city:{value}',
        'org':'org:{value}','hostname':'hostname:{value}','os':'os:{value}',
        'net':'net:{value}','product':'product:{value}','version':'version:{value}',
        'supports':['port','country','city','org','hostname','os','net','product','version','exact'],
    },
    'Internet Archive': {
        'group':'Specialist','privacy':'🟢 High','index':'Web / media archive',
        'note':'Wayback Machine + 835B+ pages archived. mediatype: filters texts/audio/movies/software/web.',
        'docs':'https://archive.org/advancedsearch.php',
        'base_url':'https://archive.org/search?query=',
        'ia_mediatype':'mediatype:{value}','ia_subject':'subject:{value}',
        'ia_creator':'creator:{value}','ia_collection':'collection:{value}',
        'ia_language':'language:{value}','ia_format':'format:{value}',
        'supports':['ia_mediatype','ia_subject','ia_creator','ia_collection',
                    'ia_language','ia_format','ia_daterange','NOT','OR','exact'],
    },

    'YouTube': {
        'group':'Social Media','privacy':'🔴 Low','index':'YouTube videos',
        'note':'No text operators. Filters (type, date, duration, sort) appended as URL params.',
        'docs':'https://support.google.com/youtube/answer/111997',
        'base_url':'https://www.youtube.com/results?search_query=',
        'supports':['exact','yt_type','yt_date','yt_duration','yt_sort'],
    },
    'Reddit': {
        'group':'Social Media','privacy':'🟡 Medium','index':'Reddit posts',
        'note':'Rich operator support in query string. sort: and time: appended as URL params.',
        'docs':'https://support.reddithelp.com/hc/en-us/articles/19696541895316-Available-search-features',
        'base_url':'https://www.reddit.com/search/?q=',
        'rd_subreddit':'subreddit:{value}','rd_author':'author:{value}','rd_site':'site:{value}',
        'rd_title':'title:{value}','rd_selftext':'selftext:{value}','rd_flair':'flair:{value}',
        'rd_nsfw':'nsfw:{value}','rd_url':'url:{value}',
        'supports':['rd_subreddit','rd_author','rd_site','rd_title','rd_selftext',
                    'rd_flair','rd_nsfw','rd_url','rd_sort','rd_time','NOT','OR','exact'],
    },
    'Twitter/X': {
        'group':'Social Media','privacy':'🔴 Low','index':'Tweets',
        'note':'Advanced search operators work directly in the query string.',
        'docs':'https://help.twitter.com/en/using-x/x-advanced-search',
        'base_url':'https://twitter.com/search?src=typed_query&q=',
        'tw_from':'from:{value}','tw_to':'to:{value}','tw_at':'@{value}',
        'tw_hashtag':'#{value}','tw_since':'since:{value}','tw_until':'until:{value}',
        'tw_lang':'lang:{value}','tw_min_rt':'min_retweets:{value}','tw_min_fav':'min_faves:{value}',
        'supports':['tw_from','tw_to','tw_at','tw_hashtag','tw_since','tw_until',
                    'tw_lang','tw_min_rt','tw_min_fav','tw_filter','NOT','OR','exact'],
    },
    'Instagram': {
        'group':'Social Media','privacy':'🔴 Low','index':'Profiles / hashtags',
        'note':'No search operators. Searches accounts, hashtags and places by plain text.',
        'docs':'',
        'base_url':'https://www.instagram.com/explore/search/?q=',
        'supports':['exact'],
    },
    'TikTok': {
        'group':'Social Media','privacy':'🔴 Low','index':'TikTok videos',
        'note':'No search operators. Plain text search only.',
        'docs':'',
        'base_url':'https://www.tiktok.com/search?q=',
        'supports':['exact'],
    },
    'VK': {
        'group':'Social Media','privacy':'🔴 Low','index':'VK posts / people',
        'note':'Russian social network. Section filter as URL param. No search syntax docs.',
        'docs':'',
        'base_url':'https://vk.com/search?q=',
        'supports':['vk_type','exact'],
    },
    'LinkedIn': {
        'group':'Social Media','privacy':'🔴 Low','index':'Profiles / jobs / posts',
        'note':'Login required for most results. Advanced filters via UI only.',
        'docs':'https://www.linkedin.com/help/linkedin/answer/302',
        'base_url':'https://www.linkedin.com/search/results/all/?keywords=',
        'supports':['exact'],
    },

    'arXiv': {
        'group':'Academic','privacy':'🟢 High','index':'Preprints (CS/Math/Physics/Bio/Econ)',
        'note':'Field prefixes: ti: au: abs: cat: co: jr: — Boolean: AND OR ANDNOT (uppercase).',
        'docs':'https://info.arxiv.org/help/find.html',
        'base_url':'https://arxiv.org/search/?searchtype=all&query=',
        'ax_ti':'ti:{value}','ax_au':'au:{value}','ax_abs':'abs:{value}',
        'ax_cat':'cat:{value}','ax_co':'co:{value}','ax_jr':'jr:{value}',
        'supports':['ax_ti','ax_au','ax_abs','ax_cat','ax_co','ax_jr','ax_daterange','NOT','OR','ANDNOT','exact'],
    },
    'Google Scholar': {
        'group':'Academic','privacy':'🔴 Low','index':'Academic papers / citations',
        'note':'author: and intitle: work in query string. Year range as URL params.',
        'docs':'https://scholar.google.com/intl/en/scholar/help.html',
        'base_url':'https://scholar.google.com/scholar?q=',
        'gs_author':'author:{value}','gs_intitle':'intitle:{value}',
        'supports':['gs_author','gs_intitle','gs_yearrange','NOT','OR','exact'],
    },
    'PubMed': {
        'group':'Academic','privacy':'🟢 High','index':'Biomedical literature (MEDLINE)',
        'note':'Rich field tags in brackets: [Title] [Author] [Journal] [MeSH Terms] etc.',
        'docs':'https://pubmed.ncbi.nlm.nih.gov/help/',
        'base_url':'https://pubmed.ncbi.nlm.nih.gov/?term=',
        'pm_title':'{value}[Title]','pm_author':'{value}[Author]',
        'pm_journal':'{value}[Journal]','pm_mesh':'{value}[MeSH Terms]',
        'pm_affil':'{value}[Affiliation]','pm_pubtype':'{value}[Publication Type]',
        'supports':['pm_title','pm_author','pm_journal','pm_mesh','pm_affil',
                    'pm_pubtype','pm_daterange','NOT','OR','exact'],
    },
    'Semantic Scholar': {
        'group':'Academic','privacy':'🟢 High','index':'Academic papers (AI-powered)',
        'note':'AI-powered semantic search. No field operators — filter by year/field via UI.',
        'docs':'https://www.semanticscholar.org/faq',
        'base_url':'https://www.semanticscholar.org/search?q=',
        'supports':['exact','NOT','OR'],
    },
    'BASE': {
        'group':'Academic','privacy':'🟢 High','index':'Open access (300M+ docs)',
        'note':'Bielefeld Academic Search Engine. ddc: Dewey subject; type: document type.',
        'docs':'https://www.base-search.net/about/en/',
        'base_url':'https://www.base-search.net/Search/Results?lookfor=',
        'ba_ddc':'ddc:{value}','ba_type':'type:{value}',
        'ba_lang':'lang:{value}','ba_coll':'collection:{value}',
        'supports':['ba_ddc','ba_type','ba_lang','ba_coll','NOT','OR','exact'],
    },
    'CORE': {
        'group':'Academic','privacy':'🟢 High','index':'Open access research',
        'note':'Aggregates open-access papers worldwide. Plain text — no public search syntax docs.',
        'docs':'',
        'base_url':'https://core.ac.uk/search?q=',
        'supports':['exact','NOT','OR'],
    },
    'Academia.edu': {
        'group':'Academic','privacy':'🟡 Medium','index':'Academic papers (user-uploaded)',
        'note':'Social platform for sharing papers. No query operators — plain text only.',
        'docs':'',
        'base_url':'https://www.academia.edu/search?q=',
        'supports':['exact'],
    },
    'ResearchGate': {
        'group':'Academic','privacy':'🟡 Medium','index':'Academic papers / profiles',
        'note':'Social network for researchers. Login recommended. No query operators.',
        'docs':'',
        'base_url':'https://www.researchgate.net/search?q=',
        'supports':['exact'],
    },
    'JSTOR': {
        'group':'Academic','privacy':'🟡 Medium','index':'Academic journals / books / images',
        'note':'Large journal archive + Artstor images (merged 2024). ti: au: field syntax supported.',
        'docs':'https://support.jstor.org/hc/en-us/articles/115004579767',
        'base_url':'https://www.jstor.org/action/doBasicSearch?Query=',
        'js_ti':'ti:{value}','js_au':'au:{value}',
        'supports':['js_ti','js_au','NOT','OR','exact'],
    },
    'OpenAlex': {
        'group':'Academic','privacy':'🟢 High','index':'Scholarly works (250M+, fully open)',
        'note':'Free replacement for Scopus/WoS. Title/author/institution/concept filters as URL params.',
        'docs':'https://developers.openalex.org/guides/searching',
        'base_url':'https://openalex.org/works?search=',
        'oax_title':'title.search:{value}','oax_author':'author.search:{value}',
        'oax_inst':'institutions.display_name.search:{value}',
        'oax_concept':'concepts.display_name.search:{value}',
        'supports':['oax_title','oax_author','oax_inst','oax_concept','oax_yearrange','exact','OR'],
    },
    'EuropePMC': {
        'group':'Academic','privacy':'🟢 High','index':'Biomedical + life sciences (Europe)',
        'note':'European PubMed Central. Richer than PubMed: full-text search, preprints, grants.',
        'docs':'https://europepmc.org/help',
        'base_url':'https://europepmc.org/search?query=',
        'ep_title':'TITLE:{value}','ep_auth':'AUTH:{value}',
        'ep_journal':'JOURNAL:{value}','ep_mesh':'MESH:{value}',
        'ep_doi':'DOI:{value}','ep_src':'SRC:{value}',
        'supports':['ep_title','ep_auth','ep_journal','ep_mesh','ep_doi','ep_src',
                    'ep_yearrange','NOT','OR','exact'],
    },
    'DOAJ': {
        'group':'Academic','privacy':'🟢 High','index':'Open access journals & articles',
        'note':'Directory of Open Access Journals — find OA journals and articles. No text operator docs.',
        'docs':'',
        'base_url':'https://doaj.org/search/articles?search=',
        'supports':['exact','NOT','OR'],
    },
    'bioRxiv': {
        'group':'Academic','privacy':'🟢 High','index':'Biology preprints',
        'note':'Cold Spring Harbor preprint server for biology. AND/OR/NOT + wildcards supported.',
        'docs':'https://www.biorxiv.org/content/search-tips',
        'base_url':'https://www.biorxiv.org/search/',
        'supports':['exact','NOT','OR'],
    },
    'NASA ADS': {
        'group':'Academic','privacy':'🟢 High','index':'Astronomy & astrophysics literature',
        'note':'NASA Astrophysics Data System. Rich Solr operators: title: author: abs: year: bibcode:.',
        'docs':'https://ui.adsabs.harvard.edu/help/search/',
        'base_url':'https://ui.adsabs.harvard.edu/search/q=',
        'ads_title':'title:{value}','ads_author':'author:{value}',
        'ads_abs':'abs:{value}','ads_year':'year:{value}',
        'ads_bibcode':'bibcode:{value}','ads_doctype':'doctype:{value}',
        'ads_property':'property:{value}',
        'supports':['ads_title','ads_author','ads_abs','ads_year','ads_bibcode',
                    'ads_doctype','ads_property','NOT','OR','exact'],
    },
    'SSRN': {
        'group':'Academic','privacy':'🟡 Medium','index':'Social science / econ / law preprints',
        'note':'Social Science Research Network. Largest preprint server for social sciences. No syntax docs.',
        'docs':'',
        'base_url':'https://papers.ssrn.com/sol3/search.taf?requestTimeout=50000&q=',
        'supports':['exact'],
    },

    'Europeana': {
        'group':'Art & Design','privacy':'🟢 High','index':'European cultural heritage',
        'note':'50M+ artworks, photos, books, music from 4000+ European institutions. Fully open.',
        'docs':'https://www.europeana.eu/en/help/search-tips',
        'base_url':'https://www.europeana.eu/en/search?query=',
        'eu_TYPE':'TYPE:{value}',      # IMAGE VIDEO TEXT SOUND 3D
        'eu_COUNTRY':'COUNTRY:{value}',
        'eu_PROVIDER':'DATA_PROVIDER:{value}',
        'eu_RIGHTS':'RIGHTS:{value}',  # *open* *restricted* *permission*
        'supports':['eu_TYPE','eu_COUNTRY','eu_PROVIDER','eu_RIGHTS','exact','NOT','OR'],
    },
    'MoMA': {
        'group':'Art & Design','privacy':'🟢 High','index':'MoMA collection',
        'note':'Museum of Modern Art, NY. 200k+ artworks. Search by artist, medium, year.',
        'docs':'https://www.moma.org/collection/',
        'base_url':'https://www.moma.org/collection/?q=',
        'supports':['exact'],
    },
    'V&A': {
        'group':'Art & Design','privacy':'🟢 High','index':'V&A collection',
        'note':'Victoria & Albert Museum, London. 1.5M+ objects — design, fashion, ceramics, textiles.',
        'docs':'https://collections.vam.ac.uk/search/',
        'base_url':'https://collections.vam.ac.uk/search/?q=',
        'supports':['exact'],
    },
    'Rijksmuseum': {
        'group':'Art & Design','privacy':'🟢 High','index':'Rijksmuseum collection',
        'note':'Dutch national museum. 700k+ objects — old masters, decorative arts, history.',
        'docs':'https://www.rijksmuseum.nl/en/search',
        'base_url':'https://www.rijksmuseum.nl/en/search?q=',
        'supports':['exact'],
    },
    'Cooper Hewitt': {
        'group':'Art & Design','privacy':'🟢 High','index':'Design museum collection',
        'note':'Smithsonian Design Museum. 215k+ design objects, drawings, prints. Open access.',
        'docs':'https://collection.cooperhewitt.org/',
        'base_url':'https://collection.cooperhewitt.org/search/?q=',
        'supports':['exact'],
    },
    'Google Arts & Culture': {
        'group':'Art & Design','privacy':'🔴 Low','index':'Global art collections',
        'note':'2000+ museums and archives worldwide. High-res image browsing.',
        'docs':'https://artsandculture.google.com/',
        'base_url':'https://artsandculture.google.com/search?q=',
        'supports':['exact'],
    },
    'Behance': {
        'group':'Art & Design','privacy':'🔴 Low','index':'Design portfolios',
        'note':'Adobe creative platform. Contemporary design work, illustration, photography. Adobe login.',
        'docs':'',
        'base_url':'https://www.behance.net/search/projects?search=',
        'supports':['exact'],
    },
    'Artsy': {
        'group':'Art & Design','privacy':'🟡 Medium','index':'Contemporary art market',
        'note':'500k+ artworks from galleries and museums. Focus on contemporary and modern art.',
        'docs':'https://www.artsy.net/articles',
        'base_url':'https://www.artsy.net/search?query=',
        'supports':['exact'],
    },
    'Tate': {
        'group':'Art & Design','privacy':'🟢 High','index':'Tate collection',
        'note':'Four Tate museums (Modern / Britain / Liverpool / St Ives). 70k+ artworks, open access.',
        'docs':'https://www.tate.org.uk/art/artworks',
        'base_url':'https://www.tate.org.uk/search?q=',
        'supports':['exact'],
    },
    'The Met': {
        'group':'Art & Design','privacy':'🟢 High','index':'Met collection',
        'note':'Metropolitan Museum of Art, NY. 490k+ open-access images. Largest US collection.',
        'docs':'https://www.metmuseum.org/art/collection',
        'base_url':'https://www.metmuseum.org/art/collection/search#!?q=',
        'supports':['exact'],
    },
    'Getty Museum': {
        'group':'Art & Design','privacy':'🟢 High','index':'Getty collection',
        'note':'J. Paul Getty Museum, Los Angeles. 290k+ artworks and photographs. Open Content initiative.',
        'docs':'https://www.getty.edu/art/collection/',
        'base_url':'https://www.getty.edu/art/collection/search?query=',
        'supports':['exact'],
    },
    'Louvre': {
        'group':'Art & Design','privacy':'🟡 Medium','index':'Louvre collection',
        'note':'Louvre Museum, Paris. 480k+ collection records with images. Interface in French & English.',
        'docs':'https://collections.louvre.fr/en',
        'base_url':'https://collections.louvre.fr/search?q=',
        'supports':['exact'],
    },
    'Art Institute Chicago': {
        'group':'Art & Design','privacy':'🟢 High','index':'AIC collection',
        'note':'Art Institute of Chicago. 300k+ works — strong in prints, drawings and European painting. Open access.',
        'docs':'https://www.artic.edu/collection',
        'base_url':'https://www.artic.edu/artworks?q=',
        'supports':['exact'],
    },
    'National Gallery London': {
        'group':'Art & Design','privacy':'🟢 High','index':'National Gallery collection',
        'note':'National Gallery, London. 2,300+ western European paintings, 13th–19th century.',
        'docs':'https://www.nationalgallery.org.uk/paintings/search-the-collection',
        'base_url':'https://www.nationalgallery.org.uk/paintings/search-the-collection?q=',
        'supports':['exact'],
    },
    'Smithsonian': {
        'group':'Art & Design','privacy':'🟢 High','index':'Smithsonian collections',
        'note':'Free search across all 19 Smithsonian museums and galleries. 4.5M+ digitised objects.',
        'docs':'https://collections.si.edu/',
        'base_url':'https://collections.si.edu/search/results.htm?q=',
        'supports':['exact'],
    },
    'Rhizome': {
        'group':'Media Art','privacy':'🟢 High','index':'Net & digital art archive',
        'note':'Net art, software art, digital preservation. ArtBase holds 2,000+ born-digital works.',
        'docs':'https://rhizome.org/art/artbase/',
        'base_url':'https://rhizome.org/search/?q=',
        'supports':['exact'],
    },
    'ZKM': {
        'group':'Media Art','privacy':'🟢 High','index':'Media art museum & research',
        'note':'Zentrum für Kunst und Medien, Karlsruhe. Largest media art museum worldwide.',
        'docs':'https://zkm.de/en/collection',
        'base_url':'https://zkm.de/en/search?keys=',
        'supports':['exact'],
    },
    'Ars Electronica': {
        'group':'Media Art','privacy':'🟡 Medium','index':'Festival & Prix archive',
        'note':'Prix Ars Electronica archive since 1987 — art, tech and society. 50k+ archive entries.',
        'docs':'https://archive.aec.at/',
        'base_url':'https://archive.aec.at/search?q=',
        'supports':['exact'],
    },
    'EAI': {
        'group':'Media Art','privacy':'🟢 High','index':'Video art distribution',
        'note':'Electronic Arts Intermix, NY. Premier distributor of artists\' video since 1971.',
        'docs':'https://www.eai.org/about',
        'base_url':'https://www.eai.org/titles?search=',
        'supports':['exact'],
    },
    'LIMA': {
        'group':'Media Art','privacy':'🟢 High','index':'Media art preservation (NL)',
        'note':'LIMA, Amsterdam. Preservation and presentation of media art. 10k+ works online.',
        'docs':'https://www.li-ma.nl/site/about',
        'base_url':'https://www.li-ma.nl/site/search?q=',
        'supports':['exact'],
    },
    'V2_': {
        'group':'Media Art','privacy':'🟢 High','index':'Unstable media institute',
        'note':'V2_ Institute for the Unstable Media, Rotterdam. Publications, events, artist projects.',
        'docs':'https://v2.nl/lab/about',
        'base_url':'https://v2.nl/search?q=',
        'supports':['exact'],
    },
    'transmediale': {
        'group':'Media Art','privacy':'🟡 Medium','index':'Media culture festival',
        'note':'transmediale, Berlin. Archive of works, artists and publications since 1988.',
        'docs':'https://transmediale.de/en',
        'base_url':'https://transmediale.de/?s=',
        'supports':['exact'],
    },
    'Vimeo': {
        'group':'Social Media','privacy':'🔴 Low','index':'Artist video platform',
        'note':'De-facto platform for artists\' moving image. High-quality video, curated channels.',
        'docs':'https://vimeo.com/search',
        'base_url':'https://vimeo.com/search?q=',
        'supports':['exact'],
    },
    'Flickr': {
        'group':'Social Media','privacy':'🟡 Medium','index':'Photo sharing',
        'note':'50B+ photos. Strong for CC-licensed images. Use licence and tag filters on-site.',
        'docs':'https://www.flickr.com/search/',
        'base_url':'https://www.flickr.com/search/?text=',
        'supports':['exact'],
    },



    'Kaggle': {
        'group':'Datasets','privacy':'🟡 Medium','index':'Kaggle datasets',
        'note':'Largest ML platform. Login recommended for full access. No search syntax docs.',
        'docs':'',
        'base_url':'https://www.kaggle.com/datasets?search=',
        'supports':['exact','NOT','OR'],
    },
    'Google Dataset Search': {
        'group':'Datasets','privacy':'🔴 Low','index':'Web (schema.org/Dataset)',
        'note':'Indexes structured dataset metadata across the web.',
        'docs':'https://datasetsearch.research.google.com/help',
        'base_url':'https://datasetsearch.research.google.com/search?query=',
        'supports':['exact','NOT','OR'],
    },
    'Zenodo': {
        'group':'Datasets','privacy':'🟢 High','index':'Open research outputs',
        'note':'CERN/OpenAIRE. Elasticsearch field syntax: field:value, AND, OR, date ranges.',
        'docs':'https://help.zenodo.org/guides/search/',
        'base_url':'https://zenodo.org/search?q=',
        'zn_title':'title:{value}','zn_creator':'creators.name:{value}',
        'zn_keyword':'keywords:{value}','zn_type':'resource_type.type:{value}',
        'zn_access':'access_right:{value}','zn_license':'metadata.rights.id:{value}',
        'supports':['zn_title','zn_creator','zn_keyword','zn_type',
                    'zn_access','zn_license','zn_daterange','exact','NOT','OR'],
    },
    'Hugging Face': {
        'group':'Datasets','privacy':'🟡 Medium','index':'ML datasets & models',
        'note':'Sort by downloads or likes. Task & language filters as URL params.',
        'docs':'https://huggingface.co/docs/hub/search',
        'base_url':'https://huggingface.co/datasets?search=',
        'supports':['hf_sort','exact'],
    },
    'Harvard Dataverse': {
        'group':'Datasets','privacy':'🟢 High','index':'Academic datasets',
        'note':'Largest academic data repository with field-scoped queries.',
        'docs':'https://guides.dataverse.org/en/latest/user/find-use-data.html',
        'base_url':'https://dataverse.harvard.edu/dataverse/harvard?q=',
        'dv_title':'title:{value}','dv_author':'authorName:{value}',
        'dv_keyword':'keywordValue:{value}','dv_subject':'subject:{value}',
        'supports':['dv_title','dv_author','dv_keyword','dv_subject','exact','NOT','OR'],
    },
    'OpenML': {
        'group':'Datasets','privacy':'🟢 High','index':'ML benchmark datasets',
        'note':'Open ML datasets with quality metrics. Focus on tabular / benchmark data.',
        'docs':'https://docs.openml.org/',
        'base_url':'https://www.openml.org/search?type=data&q=',
        'supports':['exact'],
    },
    'data.world': {
        'group':'Datasets','privacy':'🟡 Medium','index':'Community datasets',
        'note':'Social data platform. Good for open civic and social data. No search syntax docs.',
        'docs':'',
        'base_url':'https://data.world/search?q=',
        'supports':['exact','NOT','OR'],
    },
    'Papers with Code': {
        'group':'Datasets','privacy':'🟢 High','index':'ML paper datasets',
        'note':'Datasets linked to ML papers. Great for benchmarks and SOTA comparisons.',
        'docs':'',
        'base_url':'https://paperswithcode.com/datasets?q=',
        'supports':['exact'],
    },

    # ── Media Art ─────────────────────────────────────────────────────────────
    'Ars Electronica': {
        'group':'Media Art','privacy':'🟢 High','index':'Prix Ars Electronica archive',
        'note':'Prix Ars Electronica archive 1987–present. 15k+ projects across all categories. Linz, Austria.',
        'docs':'https://archive.aec.at/',
        'base_url':'https://archive.aec.at/?search=',
        'supports':['exact'],
    },
    'ZKM': {
        'group':'Media Art','privacy':'🟢 High','index':'ZKM works & exhibitions',
        'note':'Zentrum für Kunst und Medien, Karlsruhe. One of the largest media art institutions globally.',
        'docs':'https://zkm.de/en/collection',
        'base_url':'https://zkm.de/en/search?fulltext=',
        'supports':['exact'],
    },
    'Rhizome': {
        'group':'Media Art','privacy':'🟡 Medium','index':'ArtBase — net & digital art',
        'note':'Rhizome ArtBase: net art and digital art archive since 1999. Based at New Museum, NY.',
        'docs':'https://rhizome.org/artbase/',
        'base_url':'https://rhizome.org/search/?q=',
        'supports':['exact'],
    },
    'EAI': {
        'group':'Media Art','privacy':'🟢 High','index':'Video & media art catalogue',
        'note':'Electronic Arts Intermix, NY. Video and media art distribution & preservation since 1971.',
        'docs':'https://www.eai.org/titles',
        'base_url':'https://www.eai.org/search?q=',
        'supports':['exact'],
    },
    'LIMA': {
        'group':'Media Art','privacy':'🟢 High','index':'Dutch media art preservation',
        'note':'LIMA, Amsterdam. Media art preservation and distribution. Strong Dutch/European collection.',
        'docs':'https://www.li-ma.nl/site/about',
        'base_url':'https://www.li-ma.nl/site/search?q=',
        'supports':['exact'],
    },
    'Transmediale': {
        'group':'Media Art','privacy':'🟢 High','index':'Transmediale festival archive',
        'note':'Transmediale, Berlin. Major media art & culture festival archive since 1988.',
        'docs':'https://transmediale.de/archive',
        'base_url':'https://transmediale.de/?s=',
        'supports':['exact'],
    },
    'HEK Basel': {
        'group':'Media Art','privacy':'🟢 High','index':'HEK programme & artists',
        'note':'Haus der elektronischen Künste, Basel. Leading Swiss media art institution.',
        'docs':'https://www.hek.ch/program.html',
        'base_url':'https://www.hek.ch/?s=',
        'supports':['exact'],
    },
    'V2_': {
        'group':'Media Art','privacy':'🟢 High','index':'V2_ projects & publications',
        'note':'V2_ Institute for the Unstable Media, Rotterdam. Research, events, and publications archive.',
        'docs':'https://v2.nl/archive',
        'base_url':'https://v2.nl/search?q=',
        'supports':['exact'],
    },

}

SEARXNG_INSTANCE = 'https://searx.be'

FILETYPES = {
    'Documents':['pdf','doc','docx','odt','rtf','txt','md'],
    'Spreadsheets':['xls','xlsx','csv','ods'],
    'Presentations':['ppt','pptx','odp','key'],
    'Images':['jpg','jpeg','png','gif','svg','webp','tiff','raw'],
    'Video':['mp4','avi','mov','mkv','webm'],
    'Audio':['mp3','wav','flac','ogg','aac'],
    'Code':['py','js','ts','html','css','sh','rb','php','java','cpp'],
    'Data':['json','xml','yaml','yml','sql','geojson'],
    'Archives':['zip','tar','gz','rar','7z'],
}

ARTS_AND_SOCIAL = {'YouTube','Reddit','Twitter/X','Instagram','TikTok','VK','LinkedIn','Vimeo','Flickr'}
SPECIALIST_NO_FT = {'GitHub','GitLab','Shodan','Internet Archive'}
ACADEMIC_SET = {'arXiv','Google Scholar','PubMed','Semantic Scholar','BASE','CORE',
                'Academia.edu','ResearchGate','JSTOR','OpenAlex','EuropePMC','DOAJ',
                'bioRxiv','NASA ADS','SSRN'}
ART_DESIGN_SET = {'Europeana','MoMA','V&A','Rijksmuseum','Cooper Hewitt',
                  'Google Arts & Culture','Behance','Artsy',
                  'Tate','The Met','Getty Museum','Louvre',
                  'Art Institute Chicago','National Gallery London','Smithsonian'}
DATASET_SET = {'Kaggle','Google Dataset Search','Zenodo','Hugging Face',
               'Harvard Dataverse','OpenML','data.world','Papers with Code'}
MEDIA_ART_SET = {'Rhizome','ZKM','Ars Electronica','EAI','LIMA','V2_','transmediale','HEK Basel'}

NO_FILETYPES = ARTS_AND_SOCIAL | SPECIALIST_NO_FT | ACADEMIC_SET | ART_DESIGN_SET | DATASET_SET | MEDIA_ART_SET
NO_MODES     = NO_FILETYPES - {'Internet Archive'}

GROUP_COLOR = {
    'General':'#455a64','Privacy':'#2e7d32','Specialist':'#6a1b9a',
    'Social Media':'#c62828','Academic':'#1565c0',
    'Art & Design':'#6d4c41','Datasets':'#bf6900',
    'Media Art':'#00695c',
}
PRIVACY_BG = {'🔴 Low':'#fdecea','🟡 Medium':'#fffde7','🟢 High':'#e8f5e9'}


# ── Per-engine filetype group support ───────────────────────────────────────
# Most engines only reliably index Documents/Spreadsheets/Presentations via
# filetype:. Code, Data and Archives virtually never appear as indexed files
# on Bing-backed or independent engines. Archives (zip/tar/gz) return no
# results on any engine. Groups not in this list are hidden for that engine.
ENGINE_FT_GROUPS = {
    'Google':     ['Documents','Spreadsheets','Presentations','Code','Data',
                   'Images','Video','Audio','Archives'],
    'Bing':       ['Documents','Spreadsheets','Presentations'],
    'DuckDuckGo': ['Documents','Spreadsheets','Presentations'],
    'Yandex':     ['Documents','Spreadsheets','Presentations','Images'],
    'Baidu':      ['Documents','Spreadsheets','Presentations'],
    'Yahoo':      ['Documents','Spreadsheets','Presentations'],
    'Brave':      ['Documents','Spreadsheets','Presentations'],
    'Ecosia':     ['Documents','Spreadsheets','Presentations'],
    'Qwant':      ['Documents','Spreadsheets','Presentations'],
    'Kagi':       ['Documents','Spreadsheets','Presentations','Code','Data'],
    'SearXNG':    ['Documents','Spreadsheets','Presentations','Code','Data'],
}
DEFAULT_FT_GROUPS = ['Documents','Spreadsheets','Presentations']

# When image/video mode is active the engine already filters by content type —
# appending filetype: for the same type category breaks non-Google queries.
MODE_HIDE_FT = {
    'images': {'Images'},
    'video':  {'Video','Audio'},
    'news':   set(),
    'web':    set(),
}

# Yandex mime: values — extensions → string accepted by Yandex.
# Documents accept short strings (mime:pdf); media files need full MIME types.
YANDEX_MIME = {
    'pdf':'pdf','doc':'doc','docx':'docx','odt':'odt','rtf':'rtf',
    'txt':'txt','md':'txt',
    'xls':'xls','xlsx':'xlsx','csv':'csv','ods':'ods',
    'ppt':'ppt','pptx':'pptx','odp':'odp','key':'pptx',
    'jpg':'image/jpeg','jpeg':'image/jpeg','png':'image/png',
    'gif':'image/gif','svg':'image/svg+xml','webp':'image/webp',
    'tiff':'image/tiff',
    'mp3':'audio/mpeg','wav':'audio/wav','flac':'audio/flac',
    'ogg':'audio/ogg','aac':'audio/aac',
    'mp4':'video/mp4','avi':'video/x-msvideo',
    'mov':'video/quicktime','mkv':'video/x-matroska','webm':'video/webm',
}

print('✅ Engines loaded — 8 groups, 63 engines')


✅ Engines loaded — 8 groups, 63 engines


## Cell 3 — Dork builder

This is the core logic cell. `build_dork()` takes an engine name, a base query string, and a dict of options collected from the UI widgets. It assembles the operator tokens in the correct syntax for that engine, handles per-engine differences (e.g. `ANDNOT` for arXiv, `inbody:` for Bing, `mime:` for Yandex), appends URL suffix parameters where needed (year ranges, sort orders, Reddit time filters), and returns a list of `(label, dork_string, full_url)` tuples — one per selected filetype if any are checked.

In [3]:
def resolve_url(t): return t.replace('{instance}', SEARXNG_INSTANCE)

def build_dork(engine_name, base_query, options):
    engine    = ENGINES[engine_name]
    supported = engine.get('supports', [])

    def apply_ops(base, extra=None):
        parts = [f'"{base}"' if (options.get('exact') and 'exact' in supported) else base]
        if options.get('not_terms') and 'NOT' in supported:
            for t in [x.strip() for x in options['not_terms'].split(',') if x.strip()]:
                if engine_name in ('GitHub','GitLab','PubMed','EuropePMC'): parts.append(f'NOT {t}')
                elif engine_name == 'arXiv':                                 parts.append(f'ANDNOT {t}')
                else:                                                        parts.append(f'-{t}')
        if options.get('or_terms') and 'OR' in supported:
            ors = [x.strip() for x in options['or_terms'].split(',') if x.strip()]
            if ors: parts.append('(' + ' OR '.join(ors) + ')')
        if options.get('wildcard') and 'wildcard' in supported: parts.append('*')
        if options.get('numrange_from') and options.get('numrange_to') and 'numrange' in supported:
            parts.append(f"{options['numrange_from']}..{options['numrange_to']}")

        ALL_OPS = [
            ('site','site'),('host','host'),('inurl','inurl'),('intitle','intitle'),
            ('intext','intext'),('inbody','inbody'),('contains','contains'),
            ('before','before'),('after','after'),('imagesize','imagesize'),
            ('define','define'),('language','language'),('lang','lang'),('mime','mime'),
            ('user','user'),('org','org'),('repo','repo'),('path','path'),
            ('filename','filename'),('extension','extension'),('size','size'),
            ('gl_group','gl_group'),('gl_project','gl_project'),('gl_filename','gl_filename'),
            ('gl_path','gl_path'),('gl_extension','gl_extension'),('gl_language','gl_language'),
            ('port','port'),('country','country'),('city','city'),
            ('hostname','hostname'),('os','os'),('net','net'),('product','product'),('version','version'),
            ('ia_mediatype','ia_mediatype'),('ia_subject','ia_subject'),('ia_creator','ia_creator'),
            ('ia_collection','ia_collection'),('ia_language','ia_language'),('ia_format','ia_format'),
            ('rd_subreddit','rd_subreddit'),('rd_author','rd_author'),('rd_site','rd_site'),
            ('rd_title','rd_title'),('rd_selftext','rd_selftext'),('rd_flair','rd_flair'),
            ('rd_nsfw','rd_nsfw'),('rd_url','rd_url'),
            ('tw_from','tw_from'),('tw_to','tw_to'),('tw_at','tw_at'),('tw_hashtag','tw_hashtag'),
            ('tw_since','tw_since'),('tw_until','tw_until'),('tw_lang','tw_lang'),
            ('tw_min_rt','tw_min_rt'),('tw_min_fav','tw_min_fav'),
            ('ax_ti','ax_ti'),('ax_au','ax_au'),('ax_abs','ax_abs'),
            ('ax_cat','ax_cat'),('ax_co','ax_co'),('ax_jr','ax_jr'),
            ('gs_author','gs_author'),('gs_intitle','gs_intitle'),
            ('pm_title','pm_title'),('pm_author','pm_author'),('pm_journal','pm_journal'),
            ('pm_mesh','pm_mesh'),('pm_affil','pm_affil'),('pm_pubtype','pm_pubtype'),
            ('ba_ddc','ba_ddc'),('ba_type','ba_type'),('ba_lang','ba_lang'),('ba_coll','ba_coll'),
            ('js_ti','js_ti'),('js_au','js_au'),
            ('oax_title','oax_title'),('oax_author','oax_author'),
            ('oax_inst','oax_inst'),('oax_concept','oax_concept'),
            ('ep_title','ep_title'),('ep_auth','ep_auth'),('ep_journal','ep_journal'),
            ('ep_mesh','ep_mesh'),('ep_doi','ep_doi'),('ep_src','ep_src'),
            ('ads_title','ads_title'),('ads_author','ads_author'),('ads_abs','ads_abs'),
            ('ads_year','ads_year'),('ads_bibcode','ads_bibcode'),
            ('ads_doctype','ads_doctype'),('ads_property','ads_property'),
            ('eu_TYPE','eu_TYPE'),('eu_COUNTRY','eu_COUNTRY'),
            ('eu_PROVIDER','eu_PROVIDER'),('eu_RIGHTS','eu_RIGHTS'),
            ('zn_title','zn_title'),('zn_creator','zn_creator'),('zn_keyword','zn_keyword'),
            ('zn_type','zn_type'),('zn_access','zn_access'),('zn_license','zn_license'),
            ('dv_title','dv_title'),('dv_author','dv_author'),
            ('dv_keyword','dv_keyword'),('dv_subject','dv_subject'),
        ]
        for opt_k, op_k in ALL_OPS:
            val = options.get(opt_k,'').strip()
            if val and op_k in supported and op_k in engine:
                parts.append(engine[op_k].replace('{value}', val))

        if engine_name == 'Twitter/X' and 'tw_filter' in supported:
            for f in options.get('tw_filters',[]): parts.append(f'filter:{f}')
            if options.get('tw_no_rt'): parts.append('-filter:retweets')

        for prefix, yf_key, yt_key, fmt in [
            ('zn_daterange', 'zn_year_from', 'zn_year_to', 'publication_date:[{f} TO {t}]'),
            ('ia_daterange', 'ia_year_from', 'ia_year_to', 'date:[{f} TO {t}]'),
            ('ep_yearrange', 'ep_year_from', 'ep_year_to', 'PUB_YEAR:[{f} TO {t}]'),
        ]:
            if prefix in supported:
                yf, yt_ = options.get(yf_key,'').strip(), options.get(yt_key,'').strip()
                if yf and yt_: parts.append(fmt.format(f=yf, t=yt_))
                elif yf:       parts.append(fmt.format(f=yf, t='*'))

        if extra: parts.extend(extra)
        sep = ' AND ' if engine_name == 'arXiv' else ' '
        return sep.join(p for p in parts if p)

    mode    = options.get('search_mode','web')
    url_key = {'images':'image_search','video':'video_search','news':'news_search'}.get(mode,'base_url')
    base_url = resolve_url(engine.get(url_key, engine['base_url']))

    url_suffix = ''
    if engine_name == 'Hugging Face':
        s = options.get('hf_sort','none')
        if s != 'none': url_suffix = f'&sort={s}'
    if engine_name == 'Reddit':
        url_suffix = f"&sort={options.get('rd_sort','relevance')}&t={options.get('rd_time','all')}"
    if engine_name == 'VK':
        vk_t = options.get('vk_type_val','').strip()
        if vk_t: url_suffix = f'&c%5Bsection%5D={vk_t}'
    if engine_name == 'GitLab':
        url_suffix = f"&scope={options.get('gl_scope','blobs')}"
    if engine_name == 'Google Scholar':
        for k, param in [('gs_year_from','as_ylo'),('gs_year_to','as_yhi')]:
            v = options.get(k,'').strip()
            if v: url_suffix += f'&{param}={v}'
    if engine_name == 'PubMed':
        yf,yt_ = options.get('pm_year_from','').strip(), options.get('pm_year_to','').strip()
        if yf and yt_: url_suffix += f'&filter=years.{yf}-{yt_}'
        elif yf:       url_suffix += f'&filter=years.{yf}-{yf}'
    if engine_name == 'OpenAlex':
        yf,yt_ = options.get('oax_year_from','').strip(), options.get('oax_year_to','').strip()
        if yf and yt_: url_suffix += f'&filter=publication_year:{yf}-{yt_}'
        elif yf:       url_suffix += f'&filter=publication_year:{yf}'
    if engine_name == 'arXiv':
        yf,yt_ = options.get('ax_year_from','').strip(), options.get('ax_year_to','').strip()
        if yf or yt_:
            url_suffix += '&date-filter_by=date_range'
            if yf:  url_suffix += f'&date-date_from={yf}-01-01'
            if yt_: url_suffix += f'&date-date_to={yt_}-12-31'

    # ── Filetype handling ───────────────────────────────────────────────────
    def ft_token(ft):
        """Return the correct operator token for one file extension."""
        if engine_name == 'Yandex':
            return 'mime:' + YANDEX_MIME.get(ft, ft)
        return 'filetype:' + ft

    filetypes = options.get('filetypes', [])
    combine_ft = options.get('combine_ft', False)

    if filetypes and ('filetype' in supported or 'mime' in supported):
        if combine_ft and len(filetypes) > 1:
            # Single dork with all types combined via OR
            ft_part = '(' + ' OR '.join(ft_token(ft) for ft in filetypes) + ')'
            dork = apply_ops(base_query, extra=[ft_part])
            label = ' | '.join(ft.upper() for ft in filetypes)
            return [(label, dork, base_url + urllib.parse.quote_plus(dork) + url_suffix)]
        else:
            return [(ft.upper(),
                     (dork := apply_ops(base_query, extra=[ft_token(ft)])),
                     base_url + urllib.parse.quote_plus(dork) + url_suffix)
                    for ft in filetypes]
    else:
        dork = apply_ops(base_query)
        return [('Search', dork, base_url + urllib.parse.quote_plus(dork) + url_suffix)]

print('✅ Dork builder ready')


✅ Dork builder ready


## Cell 4 — Widgets

This cell creates every interactive element in the UI: the engine dropdown (grouped by category), the search phrase text field, the mode toggle (web / images / video / news), all operator input fields for every engine, the filetype checkbox grid, the boolean/combinator fields, and the two action buttons. Widgets are declared here but not yet displayed — layout happens in the final cell.

In [4]:
S    = {'description_width':'165px'}
LY   = widgets.Layout(width='490px')
LY_S = widgets.Layout(width='235px')

ENGINE_OPTIONS = [
    '── General ──────────────',  'Google','Bing','DuckDuckGo','Yandex','Baidu','Yahoo',
    '── Privacy ──────────────',  'Brave','Ecosia','Qwant','Kagi',
    '── Art & Design ─────────',  'Europeana','MoMA','V&A','Rijksmuseum','Cooper Hewitt',
                                  'Google Arts & Culture','Behance','Artsy',
                                  'Tate','The Met','Getty Museum','Louvre',
                                  'Art Institute Chicago','National Gallery London','Smithsonian',
    '── Media Art ────────────',  'Rhizome','ZKM','Ars Electronica','EAI',
                                  'LIMA','V2_','transmediale','HEK Basel',
    '── Datasets ─────────────',  'Kaggle','Google Dataset Search','Zenodo','Hugging Face',
                                  'Harvard Dataverse','OpenML','data.world','Papers with Code',
    '── Academic ─────────────',  'arXiv','Google Scholar','PubMed','Semantic Scholar',
                                  'BASE','CORE','Academia.edu','ResearchGate','JSTOR',
    '── Open Science ─────────',  'OpenAlex','EuropePMC','DOAJ','bioRxiv','NASA ADS','SSRN',
    '── Specialist ───────────',  'SearXNG','GitHub','GitLab','Shodan','Internet Archive',
    '── Social Media ─────────',  'YouTube','Reddit','Twitter/X','Instagram','TikTok','VK','LinkedIn',
                                  'Vimeo','Flickr',
]
SEPARATORS = {o for o in ENGINE_OPTIONS if o.startswith('──')}

w_engine      = widgets.Dropdown(options=ENGINE_OPTIONS, value='Brave', description='🔍 Engine:', style=S, layout=LY)
w_engine_info = widgets.HTML('')
w_searxng     = widgets.Text(value=SEARXNG_INSTANCE, description='🌐 SearXNG URL:',
                              placeholder='https://your-instance.example.com', style=S, layout=LY)
searxng_row   = widgets.VBox([w_searxng], layout=widgets.Layout(display='none'))
w_query  = widgets.Text(value='', description='📝 Search phrase:',
    placeholder='e.g.  textile design  /  neural networks  /  port:22 Apache', style=S, layout=LY)
w_exact  = widgets.Checkbox(value=False, description='Wrap in quotes (exact phrase match)', indent=False)
w_mode   = widgets.ToggleButtons(options=['web','images','video','news'], value='web',
    description='Mode:', style={'description_width':'50px','button_width':'80px'})

def tf(d,p=''): return widgets.Text(placeholder=p,description=d,style=S,layout=LY)
def tf_s(d,p=''): return widgets.Text(placeholder=p,description=d,style={'description_width':'100px'},layout=LY_S)
def yr(lbl,wf,wt):
    return widgets.VBox([widgets.HTML(f'<span style="font-size:11px;color:#555">{lbl}</span>'),
                         widgets.HBox([wf,wt],layout=widgets.Layout(gap='8px'))])

STD_NOTE = widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7">'
    '<b>Standard operators (where supported):</b> '
    '<code>site:</code> restrict to domain &nbsp;·&nbsp; '
    '<code>filetype:</code> restrict to extension &nbsp;·&nbsp; '
    '<code>inurl:</code> term in URL &nbsp;·&nbsp; '
    '<code>intitle:</code> term in &lt;title&gt; &nbsp;·&nbsp; '
    '<code>intext:</code> term in body text</div>')

w_site=tf('🌐 site:','restrict to domain — e.g. edu  or  example.com')
w_inurl=tf('🔗 inurl:','term inside the URL of a page')
w_intitle=tf('📌 intitle:','term in the HTML <title> of a page')
w_intext=tf('📄 intext:','term in the body text of a page')
w_inbody=tf('📄 inbody:','Bing/Ecosia equivalent of intext:')
w_filetype=tf('📎 filetype:','restrict to extension — e.g. pdf  or  docx')
w_contains=tf('📎 contains:','page links to filetype — Bing only')
w_before=tf('📅 before:','YYYY-MM-DD'); w_after=tf('📅 after:','YYYY-MM-DD')
w_imagesize=tf('🖼 imagesize:','1920x1080'); w_define=tf('📖 define:','word to define — Google only')
w_language=tf('🌍 language:','language code — e.g. de  or  en')
w_host=tf('💻 host:','exact domain match — Yandex'); w_mime=tf('📎 mime:','filetype — Yandex, e.g. pdf')
w_lang=tf('🌍 lang:','language — Yandex, e.g. ru  or  de')
w_gh_user=tf('👤 user:','GitHub username'); w_gh_org=tf('🏢 org:','GitHub organisation')
w_gh_repo=tf('📁 repo:','username/repo'); w_gh_path=tf('📂 path:','e.g. src/')
w_gh_file=tf('📄 filename:','e.g. config.yml'); w_gh_ext=tf('🔤 extension:','e.g. py')
w_gh_lang=tf('💬 language:','e.g. Python'); w_gh_size=tf('📐 size (KB):','e.g. 100')
w_gl_group=tf('🏢 group:','GitLab group'); w_gl_proj=tf('📁 project:','GitLab project path')
w_gl_file=tf('📄 filename:','e.g. Dockerfile'); w_gl_path=tf('📂 path:','e.g. app/')
w_gl_ext=tf('🔤 extension:','e.g. rb'); w_gl_lang=tf('💬 language:','e.g. Go')
w_gl_scope=widgets.Dropdown(options=[('Code (blobs)','blobs'),('Issues','issues'),
    ('Merge Requests','merge_requests'),('Commits','commits'),
    ('Wiki','wiki_blobs'),('Milestones','milestones')],
    value='blobs',description='🔭 Scope:',style=S,layout=LY)
w_sh_port=tf('🔌 port:','e.g. 22  or  443'); w_sh_country=tf('🌍 country:','ISO code, e.g. DE')
w_sh_city=tf('🏙 city:','e.g. Zurich'); w_sh_org=tf('🏢 org:','e.g. Amazon')
w_sh_host=tf('💻 hostname:','e.g. .edu'); w_sh_os=tf('🖥 os:','e.g. Linux')
w_sh_net=tf('📡 net:','CIDR, e.g. 192.168.0.0/24'); w_sh_prod=tf('📦 product:','e.g. Apache')
w_sh_ver=tf('🔢 version:','e.g. 2.4')
w_ia_media=tf('📂 mediatype:','texts / audio / movies / images / software / web')
w_ia_subj=tf('🏷 subject:','subject keyword'); w_ia_creator=tf('👤 creator:','author or creator')
w_ia_coll=tf('🗄 collection:','e.g. prelinger  or  gutenberg')
w_ia_lang=tf('🌍 language:','e.g. English  or  deu'); w_ia_fmt=tf('📎 format:','e.g. PDF  or  MP3')
w_ia_yf=tf_s('📅 Year from:','e.g. 1990'); w_ia_yt=widgets.Text(value='',placeholder='e.g. 2010',description='to:',style={'description_width':'25px'},layout=LY_S)
w_yt_type=widgets.Dropdown(options=[('Any','any'),('Video','video'),('Channel','channel'),('Playlist','playlist')],value='any',description='📹 Type:',style=S,layout=LY)
w_yt_date=widgets.Dropdown(options=[('Any time','any'),('Last hour','hour'),('Today','today'),('This week','week'),('This month','month'),('This year','year')],value='any',description='📅 Upload date:',style=S,layout=LY)
w_yt_dur=widgets.Dropdown(options=[('Any','any'),('< 4 min','short'),('4–20 min','medium'),('> 20 min','long')],value='any',description='⏱ Duration:',style=S,layout=LY)
w_yt_sort=widgets.Dropdown(options=[('Relevance','relevance'),('Upload date','upload_date'),('View count','view_count'),('Rating','rating')],value='relevance',description='🔢 Sort:',style=S,layout=LY)
w_rd_sub=tf('🏷 subreddit:','e.g. MachineLearning'); w_rd_author=tf('👤 author:','username')
w_rd_site=tf('🌐 site:','e.g. github.com'); w_rd_title=tf('📌 title:','word in post title')
w_rd_self=tf('📄 selftext:','word in post body'); w_rd_flair=tf('🏷 flair:','post flair text')
w_rd_nsfw=widgets.Dropdown(options=[('Include NSFW',''),('NSFW only','yes'),('No NSFW','no')],value='',description='🔞 NSFW:',style=S,layout=LY)
w_rd_sort=widgets.Dropdown(options=[('Relevance','relevance'),('New','new'),('Top','top'),('Comments','comments')],value='relevance',description='🔢 Sort:',style=S,layout=LY)
w_rd_time=widgets.Dropdown(options=[('All time','all'),('Past year','year'),('Past month','month'),('Past week','week'),('Past day','day'),('Past hour','hour')],value='all',description='📅 Time:',style=S,layout=LY)
w_tw_from=tf('👤 from:','tweets from this account'); w_tw_to=tf('👤 to:','replies to')
w_tw_at=tf('@ mention:','account mentioned'); w_tw_hash=tf('# hashtag:','hashtag without #')
w_tw_since=tf('📅 since:','YYYY-MM-DD'); w_tw_until=tf('📅 until:','YYYY-MM-DD')
w_tw_lang=tf('🌍 lang:','ISO code — e.g. en  or  de')
w_tw_min_rt=tf('🔁 min_retweets:','minimum retweet count'); w_tw_min_fv=tf('❤️ min_faves:','minimum like count')
w_tw_filters=widgets.SelectMultiple(options=['links','images','videos','media','replies','native_video'],rows=3,description='filter:',style=S,layout=LY)
w_tw_no_rt=widgets.Checkbox(value=False,description='Exclude retweets  (-filter:retweets)',indent=False)
w_vk_type=widgets.Dropdown(options=[('All',''),('People','people'),('Communities','communities'),('Videos','video'),('News','news'),('Market','market')],value='',description='📂 Section:',style=S,layout=LY)
ARXIV_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>arXiv field prefixes</b> (joined with AND): <code>ti:</code> title &nbsp;·&nbsp; <code>au:</code> author &nbsp;·&nbsp; <code>abs:</code> abstract &nbsp;·&nbsp; <code>cat:</code> category &nbsp;·&nbsp; <code>co:</code> comment &nbsp;·&nbsp; <code>jr:</code> journal ref<br>Boolean: <code>AND</code> <code>OR</code> <code>ANDNOT</code> — always uppercase</div>')
w_ax_ti=tf('📌 ti: (title)','word or phrase in paper title'); w_ax_au=tf('👤 au: (author)','e.g. hinton  or  "LeCun, Y"')
w_ax_abs=tf('📄 abs: (abstract)','word or phrase in abstract'); w_ax_cat=tf('🏷 cat: (category)','e.g. cs.AI  cs.LG  stat.ML  physics.hep-th')
w_ax_co=tf('💬 co: (comment)','word in paper comments'); w_ax_jr=tf('📰 jr: (journal)','journal reference')
w_ax_yf=tf_s('📅 Year from:','e.g. 2020'); w_ax_yt=widgets.Text(value='',placeholder='e.g. 2025',description='to:',style={'description_width':'25px'},layout=LY_S)
w_gs_author=tf('👤 author:','author name'); w_gs_intitle=tf('📌 intitle:','word in paper title')
w_gs_yf=tf_s('📅 Year from:','e.g. 2018'); w_gs_yt=widgets.Text(value='',placeholder='e.g. 2025',description='to:',style={'description_width':'25px'},layout=LY_S)
PUBMED_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>PubMed field tags</b> appended in brackets: <code>[Title]</code> <code>[Author]</code> <code>[Journal]</code> <code>[MeSH Terms]</code> <code>[Affiliation]</code> <code>[Publication Type]</code><br>Boolean: <code>AND</code> <code>OR</code> <code>NOT</code> — always uppercase</div>')
w_pm_title=tf('📌 [Title]:','word in article title'); w_pm_author=tf('👤 [Author]:','e.g. Smith J  or  "Smith, John"')
w_pm_journal=tf('📰 [Journal]:','journal name'); w_pm_mesh=tf('🏷 [MeSH Terms]:','e.g. Neoplasms')
w_pm_affil=tf('🏛 [Affiliation]:','institution name'); w_pm_pubtype=tf('📂 [Pub. Type]:','e.g. Clinical Trial')
w_pm_yf=tf_s('📅 Year from:','e.g. 2010'); w_pm_yt=widgets.Text(value='',placeholder='e.g. 2024',description='to:',style={'description_width':'25px'},layout=LY_S)
w_ba_ddc=tf('🔢 ddc:','Dewey Decimal — e.g. 004  or  610'); w_ba_type=tf('📂 type:','1=article  4=book  5=thesis')
w_ba_lang=tf('🌍 lang:','ISO code — e.g. eng  or  deu'); w_ba_coll=tf('🗄 collection:','collection identifier')
w_js_ti=tf('📌 ti: (title)','word in title'); w_js_au=tf('👤 au: (author)','author name')
OPENALEX_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>OpenAlex</b> — free, fully open replacement for Scopus/WoS (250M+ works). Field filters are appended as URL parameters; year range also as URL filter.</div>')
w_oax_title=tf('📌 title:','word in work title'); w_oax_author=tf('👤 author:','author display name')
w_oax_inst=tf('🏛 institution:','institution display name'); w_oax_concept=tf('🏷 concept:','concept/topic display name')
w_oax_yf=tf_s('📅 Year from:','e.g. 2015'); w_oax_yt=widgets.Text(value='',placeholder='e.g. 2024',description='to:',style={'description_width':'25px'},layout=LY_S)
EPMC_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>EuropePMC field tags</b>: <code>TITLE:</code> <code>AUTH:</code> <code>JOURNAL:</code> <code>MESH:</code> <code>DOI:</code> <code>SRC:</code> (source: MED PPR AGR PAT)<br>Date range appended as <code>PUB_YEAR:[from TO to]</code></div>')
w_ep_title=tf('📌 TITLE:','word in article title'); w_ep_auth=tf('👤 AUTH:','author last name or full')
w_ep_journal=tf('📰 JOURNAL:','journal name'); w_ep_mesh=tf('🏷 MESH:','MeSH term')
w_ep_doi=tf('🔗 DOI:','DOI identifier'); w_ep_src=tf('📂 SRC:','source — e.g. MED  PPR  AGR  PAT')
w_ep_yf=tf_s('📅 Year from:','e.g. 2010'); w_ep_yt=widgets.Text(value='',placeholder='e.g. 2024',description='to:',style={'description_width':'25px'},layout=LY_S)
ADS_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>NASA ADS field operators</b>: <code>title:</code> <code>author:</code> <code>abs:</code> <code>year:</code> <code>bibcode:</code> <code>doctype:</code> <code>property:</code><br><code>doctype</code> examples: article, eprint, inproceedings, book, thesis</div>')
w_ads_title=tf('📌 title:','word in title'); w_ads_author=tf('👤 author:','e.g. ^Einstein,A  (^ = first author)')
w_ads_abs=tf('📄 abs:','word in abstract'); w_ads_year=tf('📅 year:','e.g. 2020  or  2010-2020')
w_ads_bibcode=tf('🔗 bibcode:','ADS bibcode identifier'); w_ads_doctype=tf('📂 doctype:','e.g. article  or  eprint')
w_ads_property=tf('⚙️ property:','e.g. refereed  or  openaccess')
EUROPEANA_NOTE=widgets.HTML('<div style="font-size:11px;color:#555;margin-bottom:4px;line-height:1.7"><b>Europeana field filters</b>: <code>TYPE:</code> (IMAGE VIDEO TEXT SOUND 3D) <code>COUNTRY:</code> <code>DATA_PROVIDER:</code> <code>RIGHTS:</code> (*open* *restricted*)</div>')
w_eu_type=tf('📂 TYPE:','IMAGE / VIDEO / TEXT / SOUND / 3D'); w_eu_country=tf('🌍 COUNTRY:','e.g. france  or  germany')
w_eu_provider=tf('🏛 DATA_PROVIDER:','institution name'); w_eu_rights=tf('📜 RIGHTS:','e.g. *open*  or  *restricted*')
w_zn_title=tf('📌 title:','word in record title — Zenodo'); w_zn_creator=tf('👤 creators.name:','author — Zenodo')
w_zn_kw=tf('🏷 keywords:','keyword tag — Zenodo'); w_zn_type=tf('📂 resource_type:','e.g. dataset  or  publication — Zenodo')
w_zn_access=tf('🔓 access_right:','open / closed / embargoed — Zenodo'); w_zn_lic=tf('📜 license:','e.g. cc-by-4.0 — Zenodo')
w_zn_yf=tf_s('📅 Year from:','e.g. 2020'); w_zn_yt=widgets.Text(value='',placeholder='e.g. 2024',description='to:',style={'description_width':'25px'},layout=LY_S)
w_dv_title=tf('📌 title:','word in dataset title — Dataverse'); w_dv_author=tf('👤 authorName:','author — Dataverse')
w_dv_kw=tf('🏷 keywordValue:','keyword — Dataverse'); w_dv_subj=tf('📂 subject:','e.g. Medicine — Dataverse')
w_hf_sort=widgets.ToggleButtons(options=[('No sort','none'),('By downloads','downloads'),('By likes','likes')],value='none',description='Sort:',style={'description_width':'50px','button_width':'120px'})
w_not_terms=tf('➖ NOT terms:','comma-separated words to exclude')
w_or_terms=tf('↔ OR alternatives:','comma-separated alternatives')
w_wildcard=widgets.Checkbox(value=False,description='Append wildcard  *',indent=False)
w_nr_from=widgets.BoundedIntText(value=0,min=0,max=999999,description='🔢 Num range:',style={'description_width':'100px'},layout=widgets.Layout(width='240px'))
w_nr_to=widgets.BoundedIntText(value=0,min=0,max=999999,description='to:',style={'description_width':'30px'},layout=widgets.Layout(width='180px'))
ft_checks, ft_groups_ui_list, ft_group_boxes = {}, [], {}
for grp, exts in FILETYPES.items():
    cbs = []
    for e in exts:
        cb = widgets.Checkbox(value=False, description=e, indent=False,
                              layout=widgets.Layout(width='88px'))
        ft_checks[e] = cb; cbs.append(cb)
    vbox = widgets.VBox([
        widgets.HTML(f'<span style="font-size:11px;font-weight:700;color:#666;'
                     f'text-transform:uppercase;letter-spacing:.4px">{grp}</span>'),
        widgets.GridBox(cbs, layout=widgets.Layout(grid_template_columns='repeat(7, 90px)'))
    ], layout=widgets.Layout(margin='4px 0'))
    ft_groups_ui_list.append(vbox)
    ft_group_boxes[grp] = vbox   # keyed dict for per-engine / per-mode visibility
w_combine_ft = widgets.Checkbox(
    value=False,
    description='🔀 Combine selected types into one OR query  (e.g. filetype:pdf OR filetype:docx)',
    indent=False,
    layout=widgets.Layout(width='100%')
)
w_custom_ft=tf('✏️ Custom types:','e.g. epub, mobi, kmz')
btn_clear_ft=widgets.Button(description='✕ Clear all',button_style='',layout=widgets.Layout(width='110px',height='28px'))
def clear_ft(b):
    for cb in ft_checks.values(): cb.value=False
    w_custom_ft.value=''
btn_clear_ft.on_click(clear_ft)
out=widgets.Output()
btn_gen=widgets.Button(description='⚡ Generate Dorks',button_style='primary',layout=widgets.Layout(width='200px',height='36px'))
btn_all=widgets.Button(description='🌐 Compare All Engines',button_style='info',layout=widgets.Layout(width='235px',height='36px'))
print('✅ All widgets built')


✅ All widgets built


## Cell 5 — Dynamic visibility & engine badge

When the user selects a different engine, this cell's `refresh()` function fires. It swaps the visible operator fields for that engine, shows or hides the filetype panel and mode toggle, and re-renders the engine info badge (group colour, privacy rating, index type, and a `📖 Docs ↗` button when official documentation exists). Engines with no public search syntax documentation simply show no button — honest about the absence rather than linking to an irrelevant page.

In [5]:
def note(t): return widgets.HTML(f'<i style="font-size:11px;color:#888">{t}</i>')

FIELDS = {
    'Google':     [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,w_intext,w_before,w_after,w_imagesize,w_define],
    'Bing':       [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Bing uses <code>inbody:</code> instead of <code>intext:</code>'),w_inbody,w_contains,w_language],
    'DuckDuckGo': [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ DuckDuckGo does not natively support <code>intext:</code>')],
    'Yandex':     [STD_NOTE,w_site,w_host,w_inurl,w_intitle,note('ℹ️ Yandex uses <code>mime:</code> instead of <code>filetype:</code>'),w_mime,w_lang],
    'Baidu':      [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Baidu operators work best for Chinese-language content. No English operator docs.')],
    'Yahoo':      [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Yahoo is Bing-backed — Bing operators generally apply')],
    'Brave':      [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Brave: <code>intext:</code> partial / unreliable')],
    'Ecosia':     [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Ecosia uses Bing\'s <code>inbody:</code> instead of <code>intext:</code>. No dedicated operator docs.'),w_inbody,w_language],
    'Qwant':      [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,note('ℹ️ Qwant: <code>intext:</code> not officially documented')],
    'Kagi':       [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,w_intext,w_before,w_after],
    'SearXNG':    [STD_NOTE,w_site,w_filetype,w_inurl,w_intitle,w_intext,w_language],
    'GitHub':     [w_gh_user,w_gh_org,w_gh_repo,w_gh_path,w_gh_file,w_gh_ext,w_gh_lang,w_gh_size],
    'GitLab':     [w_gl_group,w_gl_proj,w_gl_file,w_gl_path,w_gl_ext,w_gl_lang,w_gl_scope],
    'Shodan':     [w_sh_port,w_sh_country,w_sh_city,w_sh_org,w_sh_host,w_sh_os,w_sh_net,w_sh_prod,w_sh_ver],
    'Internet Archive':[w_ia_media,w_ia_subj,w_ia_creator,w_ia_coll,w_ia_lang,w_ia_fmt,yr('📅 Date range:',w_ia_yf,w_ia_yt)],
    'YouTube':    [w_yt_type,w_yt_date,w_yt_dur,w_yt_sort],
    'Reddit':     [w_rd_sub,w_rd_author,w_rd_site,w_rd_title,w_rd_self,w_rd_flair,w_rd_nsfw,w_rd_sort,w_rd_time],
    'Twitter/X':  [w_tw_from,w_tw_to,w_tw_at,w_tw_hash,w_tw_since,w_tw_until,w_tw_lang,w_tw_min_rt,w_tw_min_fv,
                   widgets.HTML('<b style="font-size:11px">filter: types (multi-select):</b>'),w_tw_filters,w_tw_no_rt],
    'Instagram':  [note('No search operators — plain text search for accounts, hashtags and places')],
    'TikTok':     [note('No search operators — plain text only')],
    'VK':         [w_vk_type],
    'LinkedIn':   [note('No query operators — advanced filters via UI only')],
    'Vimeo':      [note('Plain text. Use Advanced Search on-site for duration / licence filters.')],
    'Flickr':     [note('Plain text. Use licence, tag and date filters on-site for CC-licensed images.')],
    'arXiv':      [ARXIV_NOTE,w_ax_ti,w_ax_au,w_ax_abs,w_ax_cat,w_ax_co,w_ax_jr,yr('📅 Submission year range:',w_ax_yf,w_ax_yt)],
    'Google Scholar':[w_gs_author,w_gs_intitle,yr('📅 Publication year range:',w_gs_yf,w_gs_yt)],
    'PubMed':     [PUBMED_NOTE,w_pm_title,w_pm_author,w_pm_journal,w_pm_mesh,w_pm_affil,w_pm_pubtype,yr('📅 Publication year range:',w_pm_yf,w_pm_yt)],
    'Semantic Scholar':[note('AI-powered semantic search — no field operators. Filter by year/field via UI.')],
    'BASE':       [w_ba_ddc,w_ba_type,w_ba_lang,w_ba_coll],
    'CORE':       [note('Aggregates open-access papers — plain text + NOT / OR. No public search syntax docs.')],
    'Academia.edu':  [note('Social paper platform — plain text only.')],
    'ResearchGate':  [note('Social research network — plain text only. Login recommended.')],
    'JSTOR':      [w_js_ti,w_js_au,note('Artstor images (2.5M+ artworks) are searchable via the same interface since 2024.')],
    'OpenAlex':   [OPENALEX_NOTE,w_oax_title,w_oax_author,w_oax_inst,w_oax_concept,yr('📅 Publication year range:',w_oax_yf,w_oax_yt)],
    'EuropePMC':  [EPMC_NOTE,w_ep_title,w_ep_auth,w_ep_journal,w_ep_mesh,w_ep_doi,w_ep_src,yr('📅 Publication year range:',w_ep_yf,w_ep_yt)],
    'DOAJ':       [note('Directory of Open Access Journals. Plain text + exact/NOT/OR. No search syntax docs page.')],
    'bioRxiv':    [note('Biology preprints. AND / OR / NOT + wildcards (*) supported in query string. See Docs ↗ for tips.')],
    'NASA ADS':   [ADS_NOTE,w_ads_title,w_ads_author,w_ads_abs,w_ads_year,w_ads_bibcode,w_ads_doctype,w_ads_property],
    'SSRN':       [note('Largest preprint server for social sciences, economics and law. Plain text only. No syntax docs.')],
    'Europeana':  [EUROPEANA_NOTE,w_eu_type,w_eu_country,w_eu_provider,w_eu_rights],
    'MoMA':       [note('Museum of Modern Art collection. Plain text search — filter by medium/year via UI.')],
    'V&A':        [note('Victoria & Albert Museum collection. Plain text — filter by period/material via UI.')],
    'Rijksmuseum':[note('Rijksmuseum Amsterdam collection. Plain text — filter by dating/object type via UI.')],
    'Cooper Hewitt':[note('Smithsonian Design Museum. Plain text — Elasticsearch-backed, no inline operators.')],
    'Google Arts & Culture':[note('2000+ museum collections. Plain text — filter by time/medium/colour via UI.')],
    'Behance':    [note('Adobe creative platform — plain text only. Adobe login required for full access.')],
    'Artsy':      [note('Contemporary art market — plain text. Filter by medium/period/gallery via UI.')],
    'Tate':       [note('Tate collection — plain text. Four sites: Modern / Britain / Liverpool / St Ives.')],
    'The Met':    [note('Metropolitan Museum of Art — plain text. 490k+ open-access works.')],
    'Getty Museum':[note('Getty Museum, LA — plain text. Use Open Content filter on their site for downloadable images.')],
    'Louvre':     [note('Louvre, Paris — plain text. French/English interface. Use their advanced filters on-site.')],
    'Art Institute Chicago':[note('Art Institute of Chicago — plain text. Strong print/drawing/European painting collection.')],
    'National Gallery London':[note('National Gallery, London — plain text. Western European paintings 13th–19th century.')],
    'Smithsonian': [note('Smithsonian — plain text. Searches all 19 Smithsonian museums at once.')],
    # ── Media Art ──────────────────────────────────────────────────────────
    'Ars Electronica':[note('Prix Ars Electronica archive 1987–present — plain text. Browse by category on-site.')],
    'ZKM':        [note('ZKM Karlsruhe — plain text. Works, exhibitions and events searchable.')],
    'Rhizome':    [note('Rhizome ArtBase — plain text. Net art and born-digital works since 1999.')],
    'EAI':        [note('Electronic Arts Intermix — plain text. Video art titles with artist and year filters on-site.')],
    'LIMA':       [note('LIMA, Amsterdam — plain text. Dutch/European media art preservation catalogue.')],
    'Transmediale':[note('Transmediale, Berlin — plain text. Festival archive: talks, artworks, publications.')],
    'HEK Basel':  [note('Haus der elektronischen Künste, Basel — plain text. Artists, exhibitions and events.')],
    'V2_':        [note('V2_ Rotterdam — plain text. Projects, publications and events archive.')],
    'Rhizome':        [note('Net & digital art archive. ArtBase: 2,000+ born-digital works. Plain text.')],
    'ZKM':            [note('Zentrum für Kunst und Medien, Karlsruhe. World\'s largest media art museum. Plain text.')],
    'Ars Electronica':[note('Prix Ars Electronica archive since 1987. 50k+ entries. Plain text.')],
    'EAI':            [note('Electronic Arts Intermix — premier video art distributor since 1971. Plain text.')],
    'LIMA':           [note('LIMA, Amsterdam — media art preservation. 10k+ works. Plain text.')],
    'V2_':            [note('V2_ Institute for the Unstable Media, Rotterdam. Plain text.')],
    'transmediale':   [note('transmediale Berlin — archive of works, artists and publications since 1988. Plain text.')],
    'Kaggle':     [note('Largest ML dataset platform — plain text + exact / NOT / OR. No search syntax docs.')],
    'Google Dataset Search':[note('Indexes schema.org/Dataset metadata — plain text')],
    'Zenodo':     [w_zn_title,w_zn_creator,w_zn_kw,w_zn_type,w_zn_access,w_zn_lic,yr('📅 Publication year range:',w_zn_yf,w_zn_yt)],
    'Hugging Face':  [w_hf_sort],
    'Harvard Dataverse':[w_dv_title,w_dv_author,w_dv_kw,w_dv_subj],
    'OpenML':        [note('Plain text search — tabular / benchmark ML datasets')],
    'data.world':    [note('Plain text + exact / NOT / OR. No search syntax docs.')],
    'Papers with Code':[note('Plain text — datasets linked to ML papers.')],
}

ops_box = widgets.VBox([])

def refresh(change=None):
    name = w_engine.value
    if name in SEPARATORS: return
    ops_box.children           = FIELDS.get(name, [])
    searxng_row.layout.display = '' if name == 'SearXNG' else 'none'
    ft_section.layout.display  = 'none' if name in NO_FILETYPES else ''
    w_mode.layout.display      = 'none' if name in NO_MODES else ''

    # ── Per-engine + per-mode filetype group visibility ───────────────────
    if name not in NO_FILETYPES:
        allowed = set(ENGINE_FT_GROUPS.get(name, DEFAULT_FT_GROUPS))
        hidden_by_mode = MODE_HIDE_FT.get(w_mode.value, set())
        for grp, vbox in ft_group_boxes.items():
            show = grp in allowed and grp not in hidden_by_mode
            vbox.layout.display = '' if show else 'none'

    if name in ENGINES:
        e  = ENGINES[name]
        gc = GROUP_COLOR.get(e['group'],'#555')
        pb = PRIVACY_BG.get(e['privacy'],'#fff')
        docs_url = e.get('docs','')
        docs_btn = (f'<a href="{docs_url}" target="_blank" '
                    f'style="background:#f5f5f5;border:1px solid #ccc;border-radius:4px;'
                    f'padding:2px 9px;font-size:11px;text-decoration:none;color:#333">'
                    f'📖 Docs ↗</a>') if docs_url else ''
        w_engine_info.value = (
            f'<div style="display:flex;gap:8px;align-items:center;margin:3px 0 5px 0;flex-wrap:wrap">'
            f'<span style="background:{gc};color:#fff;border-radius:4px;padding:2px 8px;font-size:11px;font-weight:700">{e["group"]}</span>'
            f'<span style="background:{pb};border-radius:4px;padding:2px 8px;font-size:11px">{e["privacy"]} privacy</span>'
            f'<span style="background:#f5f5f5;border-radius:4px;padding:2px 8px;font-size:11px">Index: {e["index"]}</span>'
            f'{docs_btn}'
            f'<span style="font-size:11px;color:#777;font-style:italic">{e["note"]}</span>'
            f'</div>'
        )

w_engine.observe(refresh, names='value')
w_mode.observe(refresh, names='value')   # re-filter ft groups when mode changes
print('✅ Visibility logic ready')


✅ Visibility logic ready


## Cell 6 — Callbacks

`collect_options()` reads the current state of every widget into a flat dictionary. `render_table()` converts a list of `(label, dork, url)` tuples into an HTML results table with copy-friendly monospace dork strings and clickable `Open ↗` links. `on_generate()` builds dorks for the selected engine; `on_compare_all()` loops through all engines grouped by category and renders a full cross-engine comparison.

In [6]:
def collect_options():
    global SEARXNG_INSTANCE
    SEARXNG_INSTANCE = w_searxng.value.strip().rstrip('/')
    fts=[ext for ext,cb in ft_checks.items() if cb.value]
    for t in w_custom_ft.value.split(','):
        t=t.strip().lower().lstrip('.')
        if t and t not in fts: fts.append(t)
    return dict(
        exact=w_exact.value, search_mode=w_mode.value,
        site=w_site.value, filetype=w_filetype.value, host=w_host.value,
        inurl=w_inurl.value, intitle=w_intitle.value, intext=w_intext.value,
        inbody=w_inbody.value, contains=w_contains.value,
        before=w_before.value, after=w_after.value,
        imagesize=w_imagesize.value, define=w_define.value,
        language=w_language.value, lang=w_lang.value, mime=w_mime.value,
        user=w_gh_user.value, org=w_gh_org.value, repo=w_gh_repo.value,
        path=w_gh_path.value, filename=w_gh_file.value, extension=w_gh_ext.value, size=w_gh_size.value,
        gl_group=w_gl_group.value, gl_project=w_gl_proj.value, gl_filename=w_gl_file.value,
        gl_path=w_gl_path.value, gl_extension=w_gl_ext.value, gl_language=w_gl_lang.value, gl_scope=w_gl_scope.value,
        port=w_sh_port.value, country=w_sh_country.value, city=w_sh_city.value,
        hostname=w_sh_host.value, os=w_sh_os.value, net=w_sh_net.value,
        product=w_sh_prod.value, version=w_sh_ver.value,
        ia_mediatype=w_ia_media.value, ia_subject=w_ia_subj.value, ia_creator=w_ia_creator.value,
        ia_collection=w_ia_coll.value, ia_language=w_ia_lang.value, ia_format=w_ia_fmt.value,
        ia_year_from=w_ia_yf.value, ia_year_to=w_ia_yt.value,
        yt_type=w_yt_type.value, yt_date=w_yt_date.value, yt_duration=w_yt_dur.value, yt_sort=w_yt_sort.value,
        rd_subreddit=w_rd_sub.value, rd_author=w_rd_author.value, rd_site=w_rd_site.value,
        rd_title=w_rd_title.value, rd_selftext=w_rd_self.value, rd_flair=w_rd_flair.value,
        rd_nsfw=w_rd_nsfw.value, rd_sort=w_rd_sort.value, rd_time=w_rd_time.value,
        tw_from=w_tw_from.value, tw_to=w_tw_to.value, tw_at=w_tw_at.value, tw_hashtag=w_tw_hash.value,
        tw_since=w_tw_since.value, tw_until=w_tw_until.value, tw_lang=w_tw_lang.value,
        tw_min_rt=w_tw_min_rt.value, tw_min_fav=w_tw_min_fv.value,
        tw_filters=list(w_tw_filters.value), tw_no_rt=w_tw_no_rt.value,
        vk_type_val=w_vk_type.value,
        ax_ti=w_ax_ti.value, ax_au=w_ax_au.value, ax_abs=w_ax_abs.value,
        ax_cat=w_ax_cat.value, ax_co=w_ax_co.value, ax_jr=w_ax_jr.value,
        ax_year_from=w_ax_yf.value, ax_year_to=w_ax_yt.value,
        gs_author=w_gs_author.value, gs_intitle=w_gs_intitle.value,
        gs_year_from=w_gs_yf.value, gs_year_to=w_gs_yt.value,
        pm_title=w_pm_title.value, pm_author=w_pm_author.value, pm_journal=w_pm_journal.value,
        pm_mesh=w_pm_mesh.value, pm_affil=w_pm_affil.value, pm_pubtype=w_pm_pubtype.value,
        pm_year_from=w_pm_yf.value, pm_year_to=w_pm_yt.value,
        ba_ddc=w_ba_ddc.value, ba_type=w_ba_type.value, ba_lang=w_ba_lang.value, ba_coll=w_ba_coll.value,
        js_ti=w_js_ti.value, js_au=w_js_au.value,
        oax_title=w_oax_title.value, oax_author=w_oax_author.value,
        oax_inst=w_oax_inst.value, oax_concept=w_oax_concept.value,
        oax_year_from=w_oax_yf.value, oax_year_to=w_oax_yt.value,
        ep_title=w_ep_title.value, ep_auth=w_ep_auth.value, ep_journal=w_ep_journal.value,
        ep_mesh=w_ep_mesh.value, ep_doi=w_ep_doi.value, ep_src=w_ep_src.value,
        ep_year_from=w_ep_yf.value, ep_year_to=w_ep_yt.value,
        ads_title=w_ads_title.value, ads_author=w_ads_author.value, ads_abs=w_ads_abs.value,
        ads_year=w_ads_year.value, ads_bibcode=w_ads_bibcode.value,
        ads_doctype=w_ads_doctype.value, ads_property=w_ads_property.value,
        eu_TYPE=w_eu_type.value, eu_COUNTRY=w_eu_country.value,
        eu_PROVIDER=w_eu_provider.value, eu_RIGHTS=w_eu_rights.value,
        zn_title=w_zn_title.value, zn_creator=w_zn_creator.value, zn_keyword=w_zn_kw.value,
        zn_type=w_zn_type.value, zn_access=w_zn_access.value, zn_license=w_zn_lic.value,
        zn_year_from=w_zn_yf.value, zn_year_to=w_zn_yt.value,
        dv_title=w_dv_title.value, dv_author=w_dv_author.value,
        dv_keyword=w_dv_kw.value, dv_subject=w_dv_subj.value,
        hf_sort=w_hf_sort.value,
        not_terms=w_not_terms.value, or_terms=w_or_terms.value, wildcard=w_wildcard.value,
        numrange_from=str(w_nr_from.value) if w_nr_from.value else '',
        numrange_to=str(w_nr_to.value)     if w_nr_to.value   else '',
        filetypes=fts,
        combine_ft=w_combine_ft.value,
    )


def render_table(name, results):
    e  = ENGINES[name]; gc = GROUP_COLOR.get(e['group'],'#555')
    docs_url = e.get('docs','')
    docs_link=(f' <a href="{docs_url}" target="_blank" '
               f'style="font-size:11px;color:#1565c0;text-decoration:none;">📖 docs</a>') if docs_url else ''
    hdr=(f'<div style="display:flex;align-items:center;gap:8px;margin:12px 0 4px">'
         f'<b style="font-size:14px">{name}</b>{docs_link}'
         f'<span style="background:{gc};color:#fff;border-radius:3px;padding:1px 8px;font-size:11px">{e["group"]}</span>'
         f'<span style="font-size:11px;color:#888">{e["privacy"]} · {e["index"]}</span>'
         f'</div>')
    rows=''.join(
        f'<tr>'
        f'<td style="padding:5px 10px;font-weight:600;white-space:nowrap;color:#555;font-size:12px">{lbl}</td>'
        f'<td style="padding:5px 10px;font-family:monospace;font-size:12px;word-break:break-all">{dork}</td>'
        f'<td style="padding:5px 10px;white-space:nowrap">'
        f'<a href="{url}" target="_blank" style="background:#1565c0;color:#fff;padding:3px 10px;border-radius:4px;text-decoration:none;font-size:11px">Open ↗</a></td></tr>'
        for lbl,dork,url in results)
    return (hdr+
        f'<table style="border-collapse:collapse;width:100%;background:#fafafa;border:1px solid #ddd;border-radius:6px;overflow:hidden;margin-bottom:4px">'
        f'<thead><tr style="background:#e3f2fd">'
        f'<th style="padding:6px 10px;text-align:left;font-size:11px">Type</th>'
        f'<th style="padding:6px 10px;text-align:left;font-size:11px">Query string</th>'
        f'<th style="padding:6px 10px;text-align:left;font-size:11px">Link</th>'
        f'</tr></thead><tbody>{rows}</tbody></table>')


def on_generate(b):
    out.clear_output(); name=w_engine.value; query=w_query.value.strip()
    if name in SEPARATORS:
        with out: display(HTML('<p style="color:orange">⚠️ Select an engine, not a section label.</p>')); return
    if not query:
        with out: display(HTML('<p style="color:red">⚠️ Enter a search phrase first.</p>')); return
    with out: display(HTML(render_table(name, build_dork(name, query, collect_options()))))


def on_compare_all(b):
    out.clear_output(); query=w_query.value.strip()
    if not query:
        with out: display(HTML('<p style="color:red">⚠️ Enter a search phrase first.</p>')); return
    opts=collect_options()
    group_order=['General','Privacy','Art & Design','Media Art','Datasets','Academic','Open Science','Specialist','Social Media']
    by_group={g:[] for g in group_order}
    for n,e in ENGINES.items(): by_group[e['group']].append(n)
    parts=['<div style="font-weight:700;font-size:16px;margin-bottom:10px">🌐 Cross-engine comparison — all 47 engines</div>']
    for grp in group_order:
        gc=GROUP_COLOR.get(grp,'#555')
        parts.append(f'<div style="background:{gc};color:#fff;border-radius:6px;padding:5px 12px;margin:14px 0 6px;font-size:12px;font-weight:700;letter-spacing:.5px;text-transform:uppercase">{grp}</div>')
        for name in by_group[grp]:
            try: parts.append(render_table(name, build_dork(name, query, opts)))
            except Exception as ex: parts.append(f'<p style="color:red">{name}: {ex}</p>')
    with out: display(HTML(''.join(parts)))

btn_gen.on_click(on_generate)
btn_all.on_click(on_compare_all)
print('✅ Callbacks registered')


✅ Callbacks registered


## Cell 7 — Layout & display

This final cell assembles all widget sections into a single vertical layout and calls `display(ui)` to render the interactive interface. Running this cell is what makes the UI appear. The `refresh()` call at the end initialises the operator fields for the default engine.

In [7]:
def section(title,*children):
    return widgets.VBox([
        widgets.HTML(f'<div style="font-weight:700;font-size:11px;letter-spacing:.6px;text-transform:uppercase;color:#666;margin-bottom:6px">{title}</div>'),
        *children
    ],layout=widgets.Layout(border='1px solid #e0e0e0',border_radius='8px',padding='10px 14px',margin='0 0 8px 0'))

ft_section = section(
    '📁 Filetypes — one dork per type  ·  unsupported groups hidden per engine  ·  (hidden for non-web engines)',
    *ft_groups_ui_list,
    widgets.HBox([w_custom_ft, btn_clear_ft], layout=widgets.Layout(align_items='center', gap='10px')),
    w_combine_ft,
)

ui = widgets.VBox([
    widgets.HTML('<h2 style="margin:0 0 10px">🔍 Search Dork Generator</h2>'),
    section('🔎 Engine & Query',w_engine,w_engine_info,searxng_row,w_query,w_exact,w_mode),
    section('🛠 Operator Filters',ops_box),
    ft_section,
    section('⚙️ Boolean & Combinators',w_not_terms,w_or_terms,
            widgets.HBox([w_nr_from,w_nr_to]),w_wildcard),
    widgets.HBox([btn_gen,btn_all],layout=widgets.Layout(gap='12px',margin='4px 0 12px')),
    out,
],layout=widgets.Layout(max_width='860px',padding='12px'))

refresh()
display(ui)
